# A neobank support agent — governed tools on a Snowflake MCP server

A **trusted support-ops agent** for a digital bank, not a public chatbot. It authenticates as a
locked-down `CUSTOMER_AGENT` role, so its whole world is the four tools below — real Snowflake
objects exposed through one MCP server:

| Tool | Snowflake object | Type |
|---|---|---|
| `classify_intent` | `SUPPORT.CLASSIFY_INTENT_PROC` → **fine-tuned Llama 3.1 8B** | Stored procedure |
| `search_help_articles` | `SUPPORT.SEARCH_HELP_ARTICLES` | Cortex Search |
| `get_transaction_status` | `SUPPORT.GET_TRANSACTION_STATUS` | SQL UDF |
| `file_ticket` | `SUPPORT.FILE_TICKET` (routes by intent) → `SUPPORT.TICKETS` | Stored procedure |

The star is **`classify_intent`** — a model whose decision boundary *you own*. It triages a message into
one of **77 real banking intents** (Banking77). We show below that it beats frontier models handed the
full label list: that is why a fine-tune is *necessary* here, not just cheaper.

## 1. The data

In [ ]:
-- Help-centre knowledge base (what search_help_articles searches)
SELECT ARTICLE_ID, TITLE, CATEGORY, LEFT(BODY, 90) || '...' AS BODY_PREVIEW
FROM MCP_HOL.SUPPORT.HELP_ARTICLES
ORDER BY ARTICLE_ID

In [ ]:
-- Support cases / transactions (what get_transaction_status looks up)
SELECT * FROM MCP_HOL.SUPPORT.CASES ORDER BY REF_ID

In [ ]:
-- Incident log (what file_ticket writes to)
SELECT * FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

In [ ]:
-- Routing policy: intent label -> queue / priority / SLA, for all 77 intents.
-- This is why classify_intent matters: file_ticket routes on the label, and
-- adjacent intents go to DIFFERENT teams (lost_or_stolen_card -> Cards & Fraud/P1
-- vs card_arrival -> Cards/P3). The model can't invent your queues.
SELECT QUEUE, COUNT(*) AS intents, MIN(PRIORITY) AS top_priority
FROM MCP_HOL.SUPPORT.INTENT_ROUTING
GROUP BY QUEUE ORDER BY intents DESC

## 2. Tool 1 — `classify_intent` (the fine-tune)

A **stored procedure** that calls an **in-account fine-tuned Llama 3.1 8B** trained on **Banking77**
(77 real neobank support intents, 10k labeled messages). The prompt is **just the raw customer
message** — no instructions, no label list. The task and the 77 labels live in the model's **weights**;
it emits one label like `card_arrival`.

> **Why a fine-tune, not a prompt?** Telling `card_arrival` from `card_delivery_estimate` from
> `lost_or_stolen_card` is a convention learned from thousands of tickets, not a rule you can write in
> a prompt. Hand a frontier model the full 77-label list and it still confuses adjacent intents. A
> fine-tune trained on your labeled history does not — see the head-to-head below.

> **Why a procedure, not a UDF?** A Cortex **fine-tuned** model is reachable through the managed MCP
> server only from a normal owner's-rights session — i.e. a stored procedure (`EXECUTE AS OWNER`). The
> GENERIC *function* path can't resolve a fine-tuned model. (Base models work either way.)

The model is trained ONCE, offline (long-running; kept out of Run All):
```sql
SELECT SNOWFLAKE.CORTEX.FINETUNE('CREATE',
  'MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', 'llama3.1-8b',
  $$SELECT TEXT AS prompt, LABEL AS completion FROM MCP_HOL.SUPPORT.B77_TRAIN$$,
  $$SELECT TEXT AS prompt, LABEL AS completion FROM MCP_HOL.SUPPORT.B77_PROBE$$);
```

**Agent input** (`input_schema` exposed by the MCP server):
```json
{ "message": "string — the customer's message" }
```
**Agent sees back:** one intent label — e.g. `card_arrival`.

In [ ]:
-- The MCP tool wrapper: a stored procedure so the fine-tuned model is reachable
-- through the managed MCP server. Bare message in -> one Banking77 label out.
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC(MESSAGE STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS OWNER
AS $$
def run(session, message):
    row = session.sql(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE('MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', ?)",
        params=[message or '']).collect()
    raw = (row[0][0] or '').strip().lower()
    import re
    m = re.search(r'[a-z_]{3,}', raw)
    return m.group(0) if m else raw
$$

In [ ]:
-- input: a customer message   ->   output: one banking intent label
CALL MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC('My new debit card still has not arrived after two weeks.')

## The proof: fine-tune vs frontier (given the full 77-label list)

The cell below classifies the held-out **154-message probe** three ways: the **fine-tune** gets only
the bare message; **claude-4-sonnet** and **openai-gpt-4.1** get the **entire 77-label list** in the
prompt (the strongest honest setup for them). Exact-match scoring. The fine-tune wins by ~12 points —
because the boundary is tacit, not promptable. (Runs ~1–2 min: 154 messages × 3 models.)

In [ ]:
-- Fair benchmark: FT (bare message) vs frontier (full label list). Exact-match.
WITH g AS (
  SELECT LISTAGG(DISTINCT LABEL, ', ') WITHIN GROUP (ORDER BY LABEL) AS labels
  FROM MCP_HOL.SUPPORT.B77_TRAIN
),
base AS (
  SELECT
    LOWER(TRIM(p.LABEL)) AS truth,
    LOWER(TRIM(SNOWFLAKE.CORTEX.COMPLETE('MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', p.TEXT))) AS ft_raw,
    LOWER(TRIM(AI_COMPLETE('claude-4-sonnet',
      'You are an intent classifier for a neobank. Classify the message into exactly ONE of these 77 intents. Reply with ONLY the label (lowercase, underscores).'
      || CHR(10) || 'Intents: ' || g.labels || CHR(10) || 'Message: ' || p.TEXT))) AS sonnet_raw,
    LOWER(TRIM(AI_COMPLETE('openai-gpt-4.1',
      'You are an intent classifier for a neobank. Classify the message into exactly ONE of these 77 intents. Reply with ONLY the label (lowercase, underscores).'
      || CHR(10) || 'Intents: ' || g.labels || CHR(10) || 'Message: ' || p.TEXT))) AS gpt_raw
  FROM MCP_HOL.SUPPORT.B77_PROBE p, g
),
norm AS (
  SELECT truth,
    REGEXP_SUBSTR(ft_raw,     '[a-z_]{3,}', 1, GREATEST(REGEXP_COUNT(ft_raw,     '[a-z_]{3,}'),1)) AS ft,
    REGEXP_SUBSTR(sonnet_raw, '[a-z_]{3,}', 1, GREATEST(REGEXP_COUNT(sonnet_raw, '[a-z_]{3,}'),1)) AS sonnet,
    REGEXP_SUBSTR(gpt_raw,    '[a-z_]{3,}', 1, GREATEST(REGEXP_COUNT(gpt_raw,    '[a-z_]{3,}'),1)) AS gpt
  FROM base
)
SELECT 'fine-tuned llama3.1-8b (bare message)' AS model, ROUND(AVG(IFF(ft=truth,1,0))*100,1) AS accuracy_pct FROM norm
UNION ALL SELECT 'openai-gpt-4.1 (full 77-label list)', ROUND(AVG(IFF(gpt=truth,1,0))*100,1) FROM norm
UNION ALL SELECT 'claude-4-sonnet (full 77-label list)', ROUND(AVG(IFF(sonnet=truth,1,0))*100,1) FROM norm
ORDER BY accuracy_pct DESC

## 3. Tool 2 — `search_help_articles`

A Cortex Search service over the bank's help-centre `HELP_ARTICLES`. The agent sends a natural-language
query and gets back the most relevant articles to ground its answer.

The MCP `tools/list` entry has a `description` AND an auto-generated `inputSchema` (only `query` is
required; `columns`/`filter`/`limit` optional). The **column names live in the description** — that is
the only place the agent learns them:
```json
{
  "name": "search_help_articles",
  "description": "...Each article has BODY plus attributes ARTICLE_ID, TITLE, CATEGORY; pass any of those in the optional 'columns' arg (only BODY is returned by default), and filter on CATEGORY via 'filter'.",
  "inputSchema": { "properties": { "query": {"type":"string"}, "columns": {"type":"array"}, "filter": {"type":"object"}, "limit": {"type":"integer","default":10} }, "required": ["query"] }
}
```

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES
  ON BODY
  ATTRIBUTES ARTICLE_ID, TITLE, CATEGORY
  WAREHOUSE = COCO_WH
  TARGET_LAG = '1 hour'
  AS SELECT BODY, ARTICLE_ID, TITLE, CATEGORY
     FROM MCP_HOL.SUPPORT.HELP_ARTICLES

In [ ]:
-- What the tool runs under the hood. Only `query` is required; `limit` optional (default 10).
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES',
  '{"query":"how long does a new card take to arrive","limit":2}'
) AS output

## 4. Tool 3 — `get_transaction_status`

A SQL UDF that returns one case/transaction's status. Owner's rights — the caller runs it without
reading the `CASES` table.

**Agent input** (`input_schema` exposed by the MCP server):
```json
{ "ref_id": "string — e.g. CASE-10001" }
```
**Agent sees back:** one line — e.g. `Case CASE-10001 (New debit card delivery): Dispatched, expected by 2026-02-12 (Sent via Royal Mail 2nd class).`

In [ ]:
CREATE OR REPLACE FUNCTION MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS(REF_ID VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
WITH input AS (SELECT REF_ID AS rid)
SELECT COALESCE(
  (SELECT 'Case ' || c.REF_ID || ' (' || c.TOPIC || '): ' || c.STATUS ||
          CASE WHEN c.STATUS IN ('Dispatched','In Progress','Pending')
               THEN ', expected by ' || TO_VARCHAR(c.ETA, 'YYYY-MM-DD') ELSE '' END ||
          CASE WHEN c.DETAIL IS NOT NULL THEN ' (' || c.DETAIL || ')' ELSE '' END || '.'
   FROM MCP_HOL.SUPPORT.CASES c, input i
   WHERE c.REF_ID = i.rid),
  'No case found for ' || (SELECT rid FROM input) || '.')
$$

In [ ]:
-- input: a case reference   ->   output: one line for that case
SELECT MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS('CASE-10001') AS output

## 5. Tool 4 — `file_ticket`

A stored procedure that files a support incident and logs it to `TICKETS`. It takes the **`intent`
label from `classify_intent`** and looks up `INTENT_ROUTING` to set the queue, priority, and SLA — so
the fine-tuned model's output decides where the case goes. Self-contained, owner's rights.

**Agent input** (`input_schema` exposed by the MCP server):
```json
{ "ref_id": "string — e.g. CASE-10001",
  "issue": "string — short description",
  "intent": "string — the label from classify_intent, e.g. card_arrival" }
```
**Agent sees back:** a confirmation — e.g. `Created incident INC0001001 for case CASE-10001 — classified as card_arrival, routed to the Cards queue at priority P3 (SLA 24h).`

In [ ]:
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(REF_ID STRING, ISSUE STRING, INTENT STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS OWNER
AS $$
def run(session, ref_id, issue, intent):
    label = (intent or '').strip().lower()
    row = session.sql(
        'SELECT QUEUE, PRIORITY, SLA_HOURS FROM MCP_HOL.SUPPORT.INTENT_ROUTING WHERE INTENT = ?',
        params=[label]).collect()
    if row:
        queue, priority, sla = row[0][0], row[0][1], row[0][2]
    else:
        label, queue, priority, sla = 'unclassified', 'General Support', 'P4', 48
    seq = session.sql('SELECT MCP_HOL.SUPPORT.TICKET_SEQ.NEXTVAL').collect()[0][0]
    inc = 'INC' + str(seq).zfill(7)
    session.sql(
        'INSERT INTO MCP_HOL.SUPPORT.TICKETS '
        '(INCIDENT_NUMBER, REF_ID, ISSUE, INTENT, QUEUE, PRIORITY, STATUS, CREATED_AT) '
        "VALUES (?, ?, ?, ?, ?, ?, 'New', CURRENT_TIMESTAMP())",
        params=[inc, ref_id, issue, label, queue, priority]).collect()
    return ('Created incident ' + inc + ' for case ' + str(ref_id)
            + ' — classified as ' + label + ', routed to the ' + queue
            + ' queue at priority ' + priority + ' (SLA ' + str(sla) + 'h).')
$$

In [ ]:
-- input: case ref + issue + intent (from classify_intent)   ->   routed incident
CALL MCP_HOL.SUPPORT.FILE_TICKET('CASE-10001', 'New debit card has not arrived after two weeks', 'card_arrival')

## 6. Create the MCP server

One statement exposes the four objects above as agent tools for the external Gemini agent.

In [ ]:
CREATE OR REPLACE MCP SERVER MCP_HOL.AGENTS.MCP_HOL
FROM SPECIFICATION $$
tools:
  - name: "classify_intent"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC"
    title: "Classify customer intent"
    description: "Classify a customer's message into exactly one of 77 neobank support intents (e.g. card_arrival, lost_or_stolen_card, declined_card_payment, exchange_rate, verify_my_identity) using an in-account FINE-TUNED model. Returns one lowercase_with_underscores label. Call this FIRST to triage the message; its label is REQUIRED by file_ticket to route the case."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          message: { type: "string", description: "The customer's message to classify" }
        required: ["message"]
  - name: "search_help_articles"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES"
    title: "Search help-centre articles"
    description: "Search the bank's help-centre knowledge base for a topic (card delivery times, lost/stolen card, declined payments, transfers, exchange rates, fees, identity verification) and return the most relevant articles. Each article has BODY plus attributes ARTICLE_ID, TITLE, CATEGORY; pass any of those names in the optional 'columns' arg (only BODY is returned by default), and filter on CATEGORY via 'filter'."
  - name: "get_transaction_status"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS"
    title: "Look up a case or transaction"
    description: "Look up the status of a SINGLE support case or transaction by its reference id. Returns the topic, current status, expected date, and channel detail."
    config:
      type: "function"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          ref_id: { type: "string", description: "The case/transaction reference, e.g. CASE-10001" }
        required: ["ref_id"]
  - name: "file_ticket"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.FILE_TICKET"
    title: "Open a support incident"
    description: "File a support incident for the customer's case. Requires the intent label from classify_intent, which routes the incident to the right queue/priority. Returns the incident number and routing."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          ref_id: { type: "string", description: "The related case reference, e.g. CASE-10001" }
          issue: { type: "string", description: "Short description of the customer's problem" }
          intent: { type: "string", description: "The intent label from classify_intent (e.g. card_arrival). Used to route the incident." }
        required: ["ref_id", "issue", "intent"]
$$;